In [1]:
import pandas as pd
import os
import re
import glob
import difflib

BASE_FOLDER = r"/Users/b/Desktop/Team Project B/OneDrive_1_01-04-2026"

all_files = glob.glob(os.path.join(BASE_FOLDER, "*", "*.csv"))
print("Total files found:", len(all_files))

Total files found: 31


In [3]:
def extract_date_time(filepath):
    fname = os.path.basename(filepath)
    m = re.search(r"(\d{8})_(\d{6})", fname)
    if m:
        catalogue_date = pd.to_datetime(m.group(1), format="%Y%m%d", errors="coerce")
        catalogue_time = m.group(2)
        return catalogue_date, catalogue_time
    return pd.NaT, None


def extract_retailer(filepath):
    return os.path.basename(os.path.dirname(filepath)).strip().lower()

In [5]:
dataframes = []

for file in all_files:
    try:
        temp = pd.read_csv(file, low_memory=False)
        catalogue_date, catalogue_time = extract_date_time(file)
        retailer = extract_retailer(file)

        temp["catalogue_date"] = catalogue_date
        temp["catalogue_time"] = catalogue_time
        temp["source_file"] = os.path.basename(file)
        temp["retailer_folder"] = retailer

        dataframes.append(temp)

    except Exception as e:
        print(f"Error reading {file}: {e}")

raw_df = pd.concat(dataframes, ignore_index=True)
print("Raw combined shape:", raw_df.shape)

Error reading /Users/b/Desktop/Team Project B/OneDrive_1_01-04-2026/IGA/iga_all_products_20260208_113459.csv: No columns to parse from file
Raw combined shape: (460036, 251)


In [19]:
standardised = []

for retailer, group in raw_df.groupby("retailer_folder"):

    temp = pd.DataFrame(index=group.index)

    retailer_clean = retailer.lower().strip()

    if "aldi" in retailer_clean:
        temp["retailer"] = "Aldi"
        temp["sku"] = group.get("sku")
        temp["product_name"] = group.get("name")
        temp["brand_name"] = group.get("brand_name")
        temp["price"] = group.get("price_dollars")
        temp["category"] = group.get("category_url_from_sitemap")
        temp["selling_size"] = group.get("aldi_selling_size")

    elif "coles" in retailer_clean:
        temp["retailer"] = "Coles"
        temp["sku"] = group.get("ProductId")
        temp["product_name"] = group.get("Name")
        temp["brand_name"] = group.get("Brand")
        temp["price"] = group.get("Price_Now")
        temp["category"] = group.get("Category")
        temp["selling_size"] = group.get("Size")

    elif "iga" in retailer_clean:
        temp["retailer"] = "IGA"
        temp["sku"] = group.get("sku")
        temp["product_name"] = group.get("name")
        temp["brand_name"] = group.get("brand_name")
        temp["price"] = group.get("price_numeric")
        temp["category"] = group.get("iga_default_category")
        temp["selling_size"] = group.get("iga_unit_of_measure_size")

    elif "wool" in retailer_clean:
        temp["retailer"] = "Woolworths"
        temp["sku"] = group.get("Stockcode")
        temp["product_name"] = group.get("Name")
        temp["brand_name"] = group.get("Brand_Searched")
        temp["price"] = group.get("Price")
        temp["category"] = group.get("SapCategoryName")
        temp["selling_size"] = group.get("PackageSize")

    else:
        print("Skipped unknown retailer folder:", retailer)
        continue

    temp["catalogue_date"] = group["catalogue_date"]
    temp["catalogue_time"] = group["catalogue_time"]
    temp["source_file"] = group["source_file"]

    standardised.append(temp)

df = pd.concat(standardised, ignore_index=True)

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["product_name"] = df["product_name"].astype(str).str.strip()
df["brand_name"] = df["brand_name"].astype(str).str.lower().str.strip()
df["selling_size"] = df["selling_size"].astype(str).str.lower().str.strip()
df["category"] = df["category"].astype(str).str.strip()

print(df["retailer"].value_counts(dropna=False))
print("Standardised shape:", df.shape)
df.head()

retailer
Woolworths    238467
Coles         131318
Aldi           48175
IGA            42076
Name: count, dtype: int64
Standardised shape: (460036, 10)


,retailer,sku,product_name,brand_name,price,category,selling_size,catalogue_date,catalogue_time,source_file
0,Aldi,399100.0,Hazelnut Mini Eggs 100g,ferrero,6.49,https://www.aldi.com.au/products/easter/k/1588...,100 g,2026-03-14,215337,aldi_all_products_20260314_215337.csv
1,Aldi,399512.0,Speckled Eggs 100g,dairy fine,3.99,https://www.aldi.com.au/products/easter/k/1588...,nan,2026-03-14,215337,aldi_all_products_20260314_215337.csv
2,Aldi,399753.0,Bunny Pink 75g,kinder,6.29,https://www.aldi.com.au/products/easter/k/1588...,75 g,2026-03-14,215337,aldi_all_products_20260314_215337.csv
3,Aldi,400016.0,Bunny 120g,sweet william,8.99,https://www.aldi.com.au/products/easter/k/1588...,120 g,2026-03-14,215337,aldi_all_products_20260314_215337.csv
4,Aldi,400463.0,Bunny Party Tub 600g,trolli,7.99,https://www.aldi.com.au/products/easter/k/1588...,600 g,2026-03-14,215337,aldi_all_products_20260314_215337.csv


In [21]:
filler_words = {
    "fresh", "original", "value", "australian", "new", "each",
    "the", "and", "with", "for", "of", "in", "pack", "pk"
}

store_words = {"aldi", "coles", "woolworths", "woolies", "iga"}


def clean_name(name):
    name = str(name).lower()
    name = re.sub(r"[^\w\s]", " ", name)
    name = re.sub(r"\b\d+\s?(g|kg|mg|ml|l|pk|pack|packs|ea)\b", " ", name)
    name = re.sub(r"\b\d+\b", " ", name)

    words = [
        w for w in name.split()
        if w not in filler_words and w not in store_words
    ]

    return " ".join(words).strip()


df["clean_name"] = df["product_name"].apply(clean_name)
df = df[df["clean_name"] != ""].copy()

df[["product_name", "clean_name"]].head()

,product_name,clean_name
0,Hazelnut Mini Eggs 100g,hazelnut mini eggs
1,Speckled Eggs 100g,speckled eggs
2,Bunny Pink 75g,bunny pink
3,Bunny 120g,bunny
4,Bunny Party Tub 600g,bunny party tub


In [23]:
def create_match_key(text, top_k=3):
    tokens = str(text).split()

    # keep only meaningful words with length more than 2
    tokens = [t for t in tokens if len(t) > 2]

    # use first 3 main words only
    tokens = tokens[:top_k]

    return " ".join(tokens)


df["match_key"] = df["clean_name"].apply(create_match_key)
df = df[df["match_key"].str.strip() != ""].copy()

print("Unique match keys:", df["match_key"].nunique())
df[["product_name", "clean_name", "match_key", "retailer"]].head(20)

Unique match keys: 37345


,product_name,clean_name,match_key,retailer
0,Hazelnut Mini Eggs 100g,hazelnut mini eggs,hazelnut mini eggs,Aldi
1,Speckled Eggs 100g,speckled eggs,speckled eggs,Aldi
2,Bunny Pink 75g,bunny pink,bunny pink,Aldi
3,Bunny 120g,bunny,bunny,Aldi
4,Bunny Party Tub 600g,bunny party tub,bunny party tub,Aldi
5,Dairy Milk Mini Eggs 243g,dairy milk mini eggs,dairy milk mini,Aldi
6,Humpty Dumpty Gift Box 132g,humpty dumpty gift box,humpty dumpty gift,Aldi
7,Chocolate Hot Cross Buns 6 Pack 450g,chocolate hot cross buns,chocolate hot cross,Aldi
8,Mini Fruit Hot Cross Buns 9 Pack 450g,mini fruit hot cross buns,mini fruit hot,Aldi
9,Fruit Hot Cross Buns 6 Pack 450g,fruit hot cross buns,fruit hot cross,Aldi


In [25]:
check_matches = df.groupby("match_key").agg(
    retailer_count=("retailer", "nunique"),
    retailers=("retailer", lambda x: ", ".join(sorted(x.dropna().unique()))),
    example_products=("product_name", lambda x: " | ".join(x.astype(str).head(3)))
).reset_index()

check_matches = check_matches.sort_values("retailer_count", ascending=False)

check_matches.head(30)

,match_key,retailer_count,retailers,example_products
14494,granny smith apples,4,"Aldi, Coles, IGA, Woolworths",Granny Smith Apples 1kg | Granny Smith Apples ...
11316,energy drink,4,"Aldi, Coles, IGA, Woolworths",V Energy Drink 250ml | V Energy Drink 250ml | ...
13289,full cream milk,4,"Aldi, Coles, IGA, Woolworths",Full Cream Milk 2L | Full Cream Milk 3L | A2 F...
17623,kit kat milk,3,"Aldi, Coles, Woolworths",Kit Kat Milk Chocolate Block 160g | Kit Kat Mi...
1743,apple blackcurrant juice,3,"Aldi, Coles, Woolworths",Apple Blackcurrant Juice 6 Pack 250ml | Apple ...
2824,bananas,3,"Aldi, Coles, IGA",Bananas 160g | Bananas 160g | Bananas 160g
6055,carrots,3,"Aldi, Coles, IGA",Carrots 1kg | Carrots 1kg | Carrots 1kg
15806,honey soy garlic,3,"Aldi, Coles, Woolworths","Honey, Soy & Garlic Marinade 375g | Honey, Soy..."
11322,energy drink blue,3,"Coles, IGA, Woolworths",Energy Drink Blue Raspberry | Energy Drink Blu...
2799,banana bread slices,3,"Aldi, Coles, Woolworths",Banana Bread Slices 5 Pack 500g | Banana Bread...


In [27]:
valid_keys = check_matches[check_matches["retailer_count"] >= 2]["match_key"]

matched_df = df[df["match_key"].isin(valid_keys)].copy()
matched_df["match_group_id"] = matched_df.groupby("match_key").ngroup() + 1

print("Matched products shape:", matched_df.shape)
print(matched_df["retailer"].value_counts())

matched_df.head()

Matched products shape: (68911, 13)
retailer
Woolworths    28935
Coles         14153
IGA           13050
Aldi          12773
Name: count, dtype: int64


,retailer,sku,product_name,brand_name,price,category,selling_size,catalogue_date,catalogue_time,source_file,clean_name,match_key,match_group_id
9,Aldi,403047.0,Fruit Hot Cross Buns 6 Pack 450g,bakers life,3.99,https://www.aldi.com.au/products/easter/k/1588...,6 each,2026-03-14,215337,aldi_all_products_20260314_215337.csv,fruit hot cross buns,fruit hot cross,637
10,Aldi,403064.0,Fruitless Hot Cross Buns 6 Pack 410g,bakers life,3.99,https://www.aldi.com.au/products/easter/k/1588...,6 each,2026-03-14,215337,aldi_all_products_20260314_215337.csv,fruitless hot cross buns,fruitless hot cross,639
11,Aldi,403327.0,Rocklea Road Bunny 170g,darrell lea,6.49,https://www.aldi.com.au/products/easter/k/1588...,170 g,2026-03-14,215337,aldi_all_products_20260314_215337.csv,rocklea road bunny,rocklea road bunny,1460
15,Aldi,526213.0,Dairy Milk Hollow Bunny 150g,cadbury,7.49,https://www.aldi.com.au/products/easter/k/1588...,150 g,2026-03-14,215337,aldi_all_products_20260314_215337.csv,dairy milk hollow bunny,dairy milk hollow,448
39,Aldi,700148.0,Dairy Milk Hollow Egg 8 Pack 153g,cadbury,5.99,https://www.aldi.com.au/products/easter/k/1588...,136 g,2026-03-14,215337,aldi_all_products_20260314_215337.csv,dairy milk hollow egg,dairy milk hollow,448


In [29]:
def similarity_score(names):
    names = list(names.dropna().astype(str).unique())

    if len(names) < 2:
        return 1.0

    scores = []

    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            score = difflib.SequenceMatcher(None, names[i], names[j]).ratio()
            scores.append(score)

    return sum(scores) / len(scores)


similarity = matched_df.groupby("match_key")["clean_name"].apply(similarity_score).reset_index()
similarity = similarity.rename(columns={"clean_name": "similarity_score"})

matched_df = matched_df.merge(similarity, on="match_key", how="left")


def confidence_level(score):
    if score >= 0.80:
        return "High"
    elif score >= 0.60:
        return "Medium"
    else:
        return "Low"


matched_df["match_confidence"] = matched_df["similarity_score"].apply(confidence_level)

matched_df[["product_name", "retailer", "match_key", "similarity_score", "match_confidence"]].head()

,product_name,retailer,match_key,similarity_score,match_confidence
0,Fruit Hot Cross Buns 6 Pack 450g,Aldi,fruit hot cross,1.000000,High
1,Fruitless Hot Cross Buns 6 Pack 410g,Aldi,fruitless hot cross,0.657534,Medium
2,Rocklea Road Bunny 170g,Aldi,rocklea road bunny,1.000000,High
3,Dairy Milk Hollow Bunny 150g,Aldi,dairy milk hollow,0.706737,Medium
4,Dairy Milk Hollow Egg 8 Pack 153g,Aldi,dairy milk hollow,0.706737,Medium


In [30]:
matched_df_price = matched_df.dropna(subset=["price"]).copy()

price_summary = matched_df_price.groupby(
    ["match_group_id", "match_key", "retailer"], as_index=False
).agg(
    avg_price=("price", "mean"),
    min_price=("price", "min"),
    max_price=("price", "max"),
    observations=("price", "count"),
    example_product=("product_name", "first"),
    category=("category", "first")
)

price_summary["avg_price"] = price_summary["avg_price"].round(2)
price_summary["min_price"] = price_summary["min_price"].round(2)
price_summary["max_price"] = price_summary["max_price"].round(2)

price_summary.head()

,match_group_id,match_key,retailer,avg_price,min_price,max_price,observations,example_product,category
0,1,abbott bakery country,IGA,5.3,5.3,5.3,1,Abbott's Bakery Country Grains Bread,"[{""category"": ""Packaged Bread"", ""categoryBread..."
1,1,abbott bakery country,Woolworths,5.2,5.2,5.2,8,Abbott's Bakery Country Grains Sandwich Slice ...,PROPRIETARY BAKERY
2,2,abbott bakery farmhouse,IGA,5.3,5.3,5.3,1,Abbott's Bakery Farmhouse Wholemeal Bread,"[{""category"": ""Packaged Bread"", ""categoryBread..."
3,2,abbott bakery farmhouse,Woolworths,5.2,5.2,5.2,8,Abbott's Bakery Farmhouse Wholemeal Bread Sand...,PROPRIETARY BAKERY
4,3,abbott bakery gluten,IGA,8.0,7.9,8.4,5,Abbott's Bakery Gluten Free Rustic White,"[{""category"": ""Packaged Bread"", ""categoryBread..."


In [34]:
cheapest = price_summary.loc[
    price_summary.groupby("match_key")["avg_price"].idxmin(),
    ["match_key", "retailer", "avg_price"]
].copy()

cheapest = cheapest.rename(columns={
    "retailer": "cheapest_retailer",
    "avg_price": "cheapest_avg_price"
})

price_summary = price_summary.merge(cheapest, on="match_key", how="left")
price_summary["is_cheapest"] = price_summary["retailer"] == price_summary["cheapest_retailer"]

price_summary.head()

,match_group_id,match_key,retailer,avg_price,min_price,max_price,observations,example_product,category,cheapest_retailer,cheapest_avg_price,is_cheapest
0,1,abbott bakery country,IGA,5.3,5.3,5.3,1,Abbott's Bakery Country Grains Bread,"[{""category"": ""Packaged Bread"", ""categoryBread...",Woolworths,5.2,False
1,1,abbott bakery country,Woolworths,5.2,5.2,5.2,8,Abbott's Bakery Country Grains Sandwich Slice ...,PROPRIETARY BAKERY,Woolworths,5.2,True
2,2,abbott bakery farmhouse,IGA,5.3,5.3,5.3,1,Abbott's Bakery Farmhouse Wholemeal Bread,"[{""category"": ""Packaged Bread"", ""categoryBread...",Woolworths,5.2,False
3,2,abbott bakery farmhouse,Woolworths,5.2,5.2,5.2,8,Abbott's Bakery Farmhouse Wholemeal Bread Sand...,PROPRIETARY BAKERY,Woolworths,5.2,True
4,3,abbott bakery gluten,IGA,8.0,7.9,8.4,5,Abbott's Bakery Gluten Free Rustic White,"[{""category"": ""Packaged Bread"", ""categoryBread...",IGA,8.0,True


In [36]:
price_summary["price_volatility"] = (
    (price_summary["max_price"] - price_summary["min_price"]) / price_summary["avg_price"]
)

def volatility_level(v):
    if pd.isna(v):
        return "Review"
    elif v <= 0.001:
        return "No change"
    elif v <= 0.10:
        return "Low"
    elif v <= 0.25:
        return "Medium"
    else:
        return "High"


price_summary["volatility_level"] = price_summary["price_volatility"].apply(volatility_level)

price_summary.head()

,match_group_id,match_key,retailer,avg_price,min_price,max_price,observations,example_product,category,cheapest_retailer,cheapest_avg_price,is_cheapest,price_volatility,volatility_level
0,1,abbott bakery country,IGA,5.3,5.3,5.3,1,Abbott's Bakery Country Grains Bread,"[{""category"": ""Packaged Bread"", ""categoryBread...",Woolworths,5.2,False,0.0000,No change
1,1,abbott bakery country,Woolworths,5.2,5.2,5.2,8,Abbott's Bakery Country Grains Sandwich Slice ...,PROPRIETARY BAKERY,Woolworths,5.2,True,0.0000,No change
2,2,abbott bakery farmhouse,IGA,5.3,5.3,5.3,1,Abbott's Bakery Farmhouse Wholemeal Bread,"[{""category"": ""Packaged Bread"", ""categoryBread...",Woolworths,5.2,False,0.0000,No change
3,2,abbott bakery farmhouse,Woolworths,5.2,5.2,5.2,8,Abbott's Bakery Farmhouse Wholemeal Bread Sand...,PROPRIETARY BAKERY,Woolworths,5.2,True,0.0000,No change
4,3,abbott bakery gluten,IGA,8.0,7.9,8.4,5,Abbott's Bakery Gluten Free Rustic White,"[{""category"": ""Packaged Bread"", ""categoryBread...",IGA,8.0,True,0.0625,Low


In [38]:
def price_suggestion(row):
    if row["is_cheapest"] and row["volatility_level"] in ["No change", "Low"]:
        return "Cheapest and stable"
    elif row["is_cheapest"]:
        return "Cheapest but price changes often"
    elif row["volatility_level"] in ["No change", "Low"]:
        return f"Stable, but cheaper at {row['cheapest_retailer']}"
    else:
        return f"Not cheapest, cheaper at {row['cheapest_retailer']}"


price_summary["price_suggestion"] = price_summary.apply(price_suggestion, axis=1)

price_summary.head()

,match_group_id,match_key,retailer,avg_price,min_price,max_price,observations,example_product,category,cheapest_retailer,cheapest_avg_price,is_cheapest,price_volatility,volatility_level,price_suggestion
0,1,abbott bakery country,IGA,5.3,5.3,5.3,1,Abbott's Bakery Country Grains Bread,"[{""category"": ""Packaged Bread"", ""categoryBread...",Woolworths,5.2,False,0.0000,No change,"Stable, but cheaper at Woolworths"
1,1,abbott bakery country,Woolworths,5.2,5.2,5.2,8,Abbott's Bakery Country Grains Sandwich Slice ...,PROPRIETARY BAKERY,Woolworths,5.2,True,0.0000,No change,Cheapest and stable
2,2,abbott bakery farmhouse,IGA,5.3,5.3,5.3,1,Abbott's Bakery Farmhouse Wholemeal Bread,"[{""category"": ""Packaged Bread"", ""categoryBread...",Woolworths,5.2,False,0.0000,No change,"Stable, but cheaper at Woolworths"
3,2,abbott bakery farmhouse,Woolworths,5.2,5.2,5.2,8,Abbott's Bakery Farmhouse Wholemeal Bread Sand...,PROPRIETARY BAKERY,Woolworths,5.2,True,0.0000,No change,Cheapest and stable
4,3,abbott bakery gluten,IGA,8.0,7.9,8.4,5,Abbott's Bakery Gluten Free Rustic White,"[{""category"": ""Packaged Bread"", ""categoryBread...",IGA,8.0,True,0.0625,Low,Cheapest and stable


In [40]:
match_details = matched_df.groupby("match_key", as_index=False).agg(
    retailers_available=("retailer", lambda x: ", ".join(sorted(x.unique()))),
    retailer_count=("retailer", "nunique"),
    example_matched_products=("product_name", lambda x: " | ".join(x.astype(str).head(5))),
    similarity_score=("similarity_score", "first"),
    match_confidence=("match_confidence", "first")
)

final_output = price_summary.merge(match_details, on="match_key", how="left")

final_output = final_output[[
    "match_group_id",
    "match_key",
    "category",
    "retailer",
    "example_product",
    "avg_price",
    "min_price",
    "max_price",
    "observations",
    "cheapest_retailer",
    "cheapest_avg_price",
    "is_cheapest",
    "price_volatility",
    "volatility_level",
    "price_suggestion",
    "retailers_available",
    "retailer_count",
    "similarity_score",
    "match_confidence",
    "example_matched_products"
]]

final_output.head()

,match_group_id,match_key,category,retailer,example_product,avg_price,min_price,max_price,observations,cheapest_retailer,cheapest_avg_price,is_cheapest,price_volatility,volatility_level,price_suggestion,retailers_available,retailer_count,similarity_score,match_confidence,example_matched_products
0,1,abbott bakery country,"[{""category"": ""Packaged Bread"", ""categoryBread...",IGA,Abbott's Bakery Country Grains Bread,5.3,5.3,5.3,1,Woolworths,5.2,False,0.0000,No change,"Stable, but cheaper at Woolworths","IGA, Woolworths",2,0.782609,Medium,Abbott's Bakery Country Grains Bread | Abbott'...
1,1,abbott bakery country,PROPRIETARY BAKERY,Woolworths,Abbott's Bakery Country Grains Sandwich Slice ...,5.2,5.2,5.2,8,Woolworths,5.2,True,0.0000,No change,Cheapest and stable,"IGA, Woolworths",2,0.782609,Medium,Abbott's Bakery Country Grains Bread | Abbott'...
2,2,abbott bakery farmhouse,"[{""category"": ""Packaged Bread"", ""categoryBread...",IGA,Abbott's Bakery Farmhouse Wholemeal Bread,5.3,5.3,5.3,1,Woolworths,5.2,False,0.0000,No change,"Stable, but cheaper at Woolworths","IGA, Woolworths",2,0.803922,High,Abbott's Bakery Farmhouse Wholemeal Bread | Ab...
3,2,abbott bakery farmhouse,PROPRIETARY BAKERY,Woolworths,Abbott's Bakery Farmhouse Wholemeal Bread Sand...,5.2,5.2,5.2,8,Woolworths,5.2,True,0.0000,No change,Cheapest and stable,"IGA, Woolworths",2,0.803922,High,Abbott's Bakery Farmhouse Wholemeal Bread | Ab...
4,3,abbott bakery gluten,"[{""category"": ""Packaged Bread"", ""categoryBread...",IGA,Abbott's Bakery Gluten Free Rustic White,8.0,7.9,8.4,5,IGA,8.0,True,0.0625,Low,Cheapest and stable,"IGA, Woolworths",2,0.734794,Medium,Abbott's Bakery Gluten Free Farmhouse Wholemea...


In [42]:
final_output.to_csv("final_matched_price_comparison.csv", index=False)

print("Saved one final CSV:")
print("final_matched_price_comparison.csv")
print("Final shape:", final_output.shape)

Saved one final CSV:
final_matched_price_comparison.csv
Final shape: (3806, 20)
